### Analysis Notebook for CWEMF

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from paper_utils import (compute_metrics, nse, load_and_merge, build_contrast_df, signal_diagnostics, error_correlation_analysis)

In [ ]:
HISTORY = Path("/Users/canruso/Desktop/casanntra_archive/history")
OUTPUT = HISTORY / "output"
SAVE_DIR = HISTORY.parent / "figures" / "paper_plots"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

C1 = "#4477AA"
C2 = "#EE6677"
C3 = "#228833"
C4 = "#CCBB44"
DPI = 300

plt.rcParams.update({"axes.titlesize": 13, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 10, "figure.dpi": DPI, "font.family": "sans-serif"})

STATIONS = ["x2", "mrz", "pct", "mal", "god", "vol", "gzl", "bdl", "nsl2", "cse", "emm2", "tms", "anh", "jer", "sal", "frk", "srv", "bac", "rsl", "oh4"]

In [ ]:
V1 = str(OUTPUT / "rma_base.multi_gru2_MSCEN_RMA_v1_Trial{}")

mscen_base = load_and_merge(V1.format(1), "base")
mscen_suisun = load_and_merge(V1.format(1), "suisun")
mscen_cache = load_and_merge(V1.format(1), "cache")
mscen_ft = load_and_merge(V1.format(1), "ft")
cw1_base = load_and_merge(V1.format(2), "base")
cw1_suisun = load_and_merge(V1.format(2), "suisun")
cw10_base = load_and_merge(V1.format(3), "base")
cw10_suisun = load_and_merge(V1.format(3), "suisun")
cw100_base = load_and_merge(V1.format(4), "base")
cw100_suisun = load_and_merge(V1.format(4), "suisun")
mstage_base = load_and_merge(OUTPUT / "dsm2.rma_base_gru2_RMA-1a_MSTAGE_base_Trial1")
mstage_suisun = load_and_merge(OUTPUT / "rma_base.suisun_gru2_RMA-2a_MSTAGE_suisun_Trial1")
mstage_cache = load_and_merge(OUTPUT / "rma_base.cache_gru2_RMA-ctrl-cache_MSTAGE_cache_Trial1")
mstage_ft = load_and_merge(OUTPUT / "rma_base.ft_gru2_RMA-ctrl-ft_MSTAGE_ft_Trial1")


ctr_mscen_suisun = build_contrast_df(mscen_base, mscen_suisun, STATIONS)
ctr_mscen_cache = build_contrast_df(mscen_base, mscen_cache, STATIONS)
ctr_mscen_ft = build_contrast_df(mscen_base, mscen_ft, STATIONS)
ctr_mstage_suisun = build_contrast_df(mstage_base, mstage_suisun, STATIONS)
ctr_mstage_cache = build_contrast_df(mstage_base, mstage_cache, STATIONS)
ctr_mstage_ft = build_contrast_df(mstage_base, mstage_ft, STATIONS)
ctr_cw1 = build_contrast_df(cw1_base, cw1_suisun, STATIONS)
ctr_cw10 = build_contrast_df(cw10_base, cw10_suisun, STATIONS)
ctr_cw100 = build_contrast_df(cw100_base, cw100_suisun, STATIONS)

print("MSCEN v1T1:", len(mscen_base), "rows")
print("MSTAGE:", len(mstage_base), "rows")
print("Contrast frames:", len(ctr_mscen_suisun), "rows each")

In [ ]:
df_v1 = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v1_master_results.csv")      
df_v2 = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v2_master_results.csv")       
df_v3_2 = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v3.2_master_results.csv")   
df_v3_3 = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v3.3_master_results.csv")  
df_v4 = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v4_master_results.csv")
df_v5 = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v5_master_results.csv")
df_mstage_base = pd.read_csv(HISTORY / "gridsearch_RMA-1a_MSTAGE_base_master_results.csv")
df_mstage_suisun = pd.read_csv(HISTORY / "gridsearch_RMA-2a_MSTAGE_suisun_master_results.csv")
df_mstage_cache = pd.read_csv(HISTORY / "gridsearch_RMA-ctrl-cache_MSTAGE_cache_master_results.csv")
df_mstage_ft = pd.read_csv(HISTORY / "gridsearch_RMA-ctrl-ft_MSTAGE_ft_master_results.csv")
df_noDSM2_suisun = pd.read_csv(HISTORY / "gridsearch_RMA-noDSM2_MSTAGE_suisun_master_results.csv")
df_noDSM2_cache = pd.read_csv(HISTORY / "gridsearch_RMA-noDSM2_MSTAGE_cache_master_results.csv")
df_noDSM2_ft = pd.read_csv(HISTORY / "gridsearch_RMA-noDSM2_MSTAGE_ft_master_results.csv")

print("14 master results CSVs loaded.")

## 1. DSM2 Pretraining Effect

**Question:** Does pretraining on DSM2 (1D hydrodynamic model) before fine-tuning on RMA (2D) improve prediction quality?

Both MSCEN and MSTAGE architectures are compared with and without DSM2 pretraining. The noDSM2 experiments were given higher learning rates and 3x more training epochs to compensate - and still fell short.

In [ ]:
dsm2_rows = [
    # MSTAGE with DSM2 (lr=0.001/0.0005, 45ep total)
    {"Architecture": "MSTAGE", "Scenario": "suisun", "Config": "With DSM2 (lr=1e-3, 45ep)",
     "NSE": df_mstage_suisun.iloc[0]["mean_nse_target"]},
    {"Architecture": "MSTAGE", "Scenario": "cache", "Config": "With DSM2 (lr=1e-3, 45ep)",
     "NSE": df_mstage_cache.iloc[0]["mean_nse_target"]},
    {"Architecture": "MSTAGE", "Scenario": "ft", "Config": "With DSM2 (lr=1e-3, 45ep)",
     "NSE": df_mstage_ft.iloc[0]["mean_nse_target"]},
    # MSTAGE without DSM2 best (lr=0.008/0.003, 150ep total)
    {"Architecture": "MSTAGE", "Scenario": "suisun", "Config": "No DSM2 (lr=3e-3, 150ep)",
     "NSE": df_noDSM2_suisun.iloc[0]["mean_nse_target"]},
    {"Architecture": "MSTAGE", "Scenario": "cache", "Config": "No DSM2 (lr=3e-3, 150ep)",
     "NSE": df_noDSM2_cache.iloc[0]["mean_nse_target"]},
    {"Architecture": "MSTAGE", "Scenario": "ft", "Config": "No DSM2 (lr=3e-3, 150ep)",
     "NSE": df_noDSM2_ft.iloc[0]["mean_nse_target"]},
    # MSCEN with DSM2
    {"Architecture": "MSCEN", "Scenario": "suisun", "Config": "With DSM2 (lr=1e-3, 45ep)",
     "NSE": df_v1.iloc[0]["mean_nse_suisun"]},
    {"Architecture": "MSCEN", "Scenario": "cache", "Config": "With DSM2 (lr=1e-3, 45ep)",
     "NSE": df_v1.iloc[0]["mean_nse_cache"]},
    {"Architecture": "MSCEN", "Scenario": "ft", "Config": "With DSM2 (lr=1e-3, 45ep)",
     "NSE": df_v1.iloc[0]["mean_nse_ft"]},
    # MSCEN without DSM2 best (v3.3, lr=0.008/0.003, 150ep)
    {"Architecture": "MSCEN", "Scenario": "suisun", "Config": "No DSM2 (lr=3e-3, 150ep)",
     "NSE": df_v3_3.iloc[0]["mean_nse_suisun"]},
    {"Architecture": "MSCEN", "Scenario": "cache", "Config": "No DSM2 (lr=3e-3, 150ep)",
     "NSE": df_v3_3.iloc[0]["mean_nse_cache"]},
    {"Architecture": "MSCEN", "Scenario": "ft", "Config": "No DSM2 (lr=3e-3, 150ep)",
     "NSE": df_v3_3.iloc[0]["mean_nse_ft"]},
]

df_dsm2 = pd.DataFrame(dsm2_rows)
df_dsm2["NSE"] = df_dsm2["NSE"].round(4)
pivot_dsm2 = df_dsm2.pivot_table(index=["Architecture", "Scenario"], columns="Config", values="NSE")
pivot_dsm2["Delta"] = pivot_dsm2.iloc[:, 1] - pivot_dsm2.iloc[:, 0]  # with - without
print(pivot_dsm2.round(4).to_string())

In [ ]:
# DSM2 pretraining: 2-panel (MSTAGE | MSCEN), with vs without
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True, sharey=True)

scenarios = ['suisun', 'cache', 'ft']
x = np.arange(len(scenarios))
w = 0.35

# MSTAGE data
mstage_with = [df_mstage_suisun.iloc[0]['mean_nse_target'],
               df_mstage_cache.iloc[0]['mean_nse_target'],
               df_mstage_ft.iloc[0]['mean_nse_target']]
mstage_without = [df_noDSM2_suisun.iloc[0]['mean_nse_target'],
                  df_noDSM2_cache.iloc[0]['mean_nse_target'],
                  df_noDSM2_ft.iloc[0]['mean_nse_target']]

# MSCEN data
mscen_with = [df_v1.iloc[0]['mean_nse_suisun'],
              df_v1.iloc[0]['mean_nse_cache'],
              df_v1.iloc[0]['mean_nse_ft']]
mscen_without = [df_v3_3.iloc[0]['mean_nse_suisun'],
                 df_v3_3.iloc[0]['mean_nse_cache'],
                 df_v3_3.iloc[0]['mean_nse_ft']]

for ax, with_vals, without_vals, title in [
    (ax1, mstage_with, mstage_without, 'Independent (MSTAGE)'),
    (ax2, mscen_with, mscen_without, 'Shared Trunk (MSCEN)')]:
    ax.bar(x - w/2, with_vals, w, color=C1, edgecolor='black', lw=0.8, label='With DSM2')
    ax.bar(x + w/2, without_vals, w, color=C2, edgecolor='black', lw=0.8, label='Without DSM2')
    ax.set_xticks(x)
    ax.set_xticklabels([s.capitalize() for s in scenarios], fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0.6, 1.0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=10, loc='lower right')
    for i, (wv, wov) in enumerate(zip(with_vals, without_vals)):
        ax.text(x[i] - w/2, wv + 0.005, f'{wv:.2f}', ha='center', fontsize=10, fontweight='bold')
        ax.text(x[i] + w/2, wov + 0.005, f'{wov:.2f}', ha='center', fontsize=10, fontweight='bold')

ax1.set_ylabel('NSE', fontsize=12)
plt.savefig(SAVE_DIR / 'fig1a_dsm2_pretraining.png', dpi=DPI, bbox_inches='tight')
plt.show()

### Section 1 Experiment Log

| Label | Experiment | Steps | Key Params | Hypothesis | Result |
|-------|-----------|-------|------------|------------|--------|
| MSTAGE+DSM2 (RMA-2a) | DSM2 -> suisun | 2-step | lr=1e-3/5e-4, 45ep | Baseline with pretraining | NSE=0.935 |
| MSTAGE+DSM2 (ctrl-cache) | DSM2 -> cache | 2-step | lr=1e-3/5e-4, 45ep | Baseline with pretraining | NSE=0.949 |
| MSTAGE+DSM2 (ctrl-ft) | DSM2 -> ft | 2-step | lr=1e-3/5e-4, 45ep | Baseline with pretraining | NSE=0.950 |
| MSTAGE-noDSM2 | RMA scenario from scratch | 1-step | lr=8e-3/3e-3, 150ep | Best without pretraining | NSE=0.729/0.725/0.746 |
| MSCEN+DSM2 (v1 T1) | DSM2 -> multi-head (3 scenarios) | 2-step | lr=1e-3/5e-4, 45ep, cw=0 | Baseline with pretraining | NSE=0.936 |
| MSCEN-noDSM2 (v3) | Multi-head from scratch | 1-step | lr=1e-3/5e-4, 45ep | Same params as DSM2 run | NSE=0.211 (catastrophic) |
| MSCEN-noDSM2 (v3.1) | Multi-head from scratch | 1-step | lr=8e-3/1e-3, 65ep | Higher LR to compensate | NSE=0.501 |
| MSCEN-noDSM2 (v3.2) | Multi-head from scratch, 1-scen | 1-step | lr=8e-3/3e-3, 150ep | More epochs + higher LR | NSE=0.805 |
| MSCEN-noDSM2 (v3.3) | Multi-head from scratch, 3-scen | 1-step | lr=8e-3/3e-3, 150ep | Same as v3.2 but 3 scenarios | NSE=0.797 |

**Conclusion:** DSM2 pretraining gives +0.13-0.22 NSE. Without pretraining, even 3x more epochs and higher LR cannot close the gap. Using DSM2-tuned hyperparams without pretraining is catastrophic (v3: 0.21) - the model needs either pretraining OR aggressive LR, not both.

## 2. Absolute Prediction Quality

**Finding:** Both MSCEN and MSTAGE achieve comparable absolute NSE (~0.93-0.95) across all scenarios with DSM2 pretraining. The multi-head architecture does not sacrifice individual prediction quality.

In [ ]:
abs_rows = [
    {"Scenario": "base",
     "MSCEN": df_v1.iloc[0]["mean_nse_base"],
     "MSTAGE": df_mstage_base.iloc[0]["mean_nse_base"]},
    {"Scenario": "suisun",
     "MSCEN": df_v1.iloc[0]["mean_nse_suisun"],
     "MSTAGE": df_mstage_suisun.iloc[0]["mean_nse_target"]},
    {"Scenario": "cache",
     "MSCEN": df_v1.iloc[0]["mean_nse_cache"],
     "MSTAGE": df_mstage_cache.iloc[0]["mean_nse_target"]},
    {"Scenario": "ft",
     "MSCEN": df_v1.iloc[0]["mean_nse_ft"],
     "MSTAGE": df_mstage_ft.iloc[0]["mean_nse_target"]},
]

df_abs = pd.DataFrame(abs_rows)
df_abs["Delta"] = (df_abs["MSCEN"] - df_abs["MSTAGE"]).round(4)
df_abs["MSCEN"] = df_abs["MSCEN"].round(4)
df_abs["MSTAGE"] = df_abs["MSTAGE"].round(4)

print(df_abs.to_string(index=False))
print(f"\nMSCEN mean: {df_abs['MSCEN'].mean():.4f}")
print(f"MSTAGE mean: {df_abs['MSTAGE'].mean():.4f}")
print(f"Delta: {df_abs['Delta'].mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

scenarios = [s.capitalize() if s != 'ft' else 'Franks Tract' for s in df_abs['Scenario'].values]
x = np.arange(len(scenarios))
w = 0.3

ax.bar(x - w/2, df_abs['MSCEN'].values, w, label='MSCEN (shared trunk)', color=C1, edgecolor='black', lw=0.8)
ax.bar(x + w/2, df_abs['MSTAGE'].values, w, label='MSTAGE (independent)', color=C2, edgecolor='black', lw=0.8)

for i in range(len(scenarios)):
    for j, col in enumerate(['MSCEN', 'MSTAGE']):
        val = df_abs[col].values[i]
        xpos = x[i] + (-w/2 if j == 0 else w/2)
        ax.text(xpos, val + 0.001, f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=12)
ax.set_ylabel('NSE', fontsize=12)
ax.set_ylim(0.80, 1.0)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.savefig(SAVE_DIR / 'fig2_absolute_nse.png', dpi=DPI, bbox_inches='tight')
plt.show()

## 3. Shared Trunk Dramatically Improves Contrast Quality

**This is the central result.** While absolute prediction quality is nearly identical between MSCEN and MSTAGE (~0.93 NSE), contrast quality diverges dramatically. The shared trunk is the mechanism - not contrastive loss, not architecture complexity.

In [ ]:
contrast_table_rows = []
scenario_data = [
    ("suisun", ctr_mscen_suisun, ctr_mstage_suisun),
    ("cache", ctr_mscen_cache, ctr_mstage_cache),
    ("ft", ctr_mscen_ft, ctr_mstage_ft),
]

for scenario, ctr_ms, ctr_st in scenario_data:
    for st in STATIONS:
        tc, pc = f"{st}_true_contrast", f"{st}_pred_contrast"
        if tc not in ctr_ms.columns:
            continue
        contrast_table_rows.append({
            "scenario": scenario, "station": st,
            "MSCEN": round(nse(ctr_ms[tc].values, ctr_ms[pc].values), 4),
            "MSTAGE": round(nse(ctr_st[tc].values, ctr_st[pc].values), 4),
        })

df_contrast = pd.DataFrame(contrast_table_rows)

# Summary by scenario
summary = df_contrast.groupby("scenario")[["MSCEN", "MSTAGE"]].mean().round(4)
summary["advantage"] = (summary["MSCEN"] - summary["MSTAGE"]).round(4)
print("Mean contrast NSE by scenario:")
print(summary.to_string())

# Full suisun table
print("\n\nPer-station contrast NSE (suisun):")
suisun = df_contrast[df_contrast["scenario"] == "suisun"].sort_values("MSCEN", ascending=False)
print(suisun[["station", "MSCEN", "MSTAGE"]].to_string(index=False))

In [ ]:
# Bar chart: contrast NSE per station, MSCEN vs MSTAGE (suisun)
df_bar = suisun.copy()
fig, ax = plt.subplots(figsize=(14, 5.5), constrained_layout=True)

x = np.arange(len(df_bar))
w = 0.35
mstage_clipped = df_bar['MSTAGE'].clip(lower=0)

ax.bar(x - w/2, df_bar['MSCEN'].values, w, label='MSCEN (shared trunk)', color=C1, edgecolor='black', lw=0.8)
ax.bar(x + w/2, mstage_clipped.values, w, label='MSTAGE (independent)', color=C2, edgecolor='black', lw=0.8)

ax.set_xticks(x)
ax.set_xticklabels(df_bar['station'].values, rotation=45, ha='right', fontsize=11)
ax.set_ylabel('Contrast NSE', fontsize=12)
ax.axhline(0, color='black', lw=0.8)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1)
ax.grid(axis='y', alpha=0.3)

for i, (_, r) in enumerate(df_bar.iterrows()):
    if r['MSTAGE'] < 0:
        ax.text(x[i] + w/2, 0.02, f"{r['MSTAGE']:.1f}", ha='center', fontsize=9, color=C2, fontweight='bold', va='bottom')

plt.savefig(SAVE_DIR / 'fig3_contrast_nse_barchart.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side contrast time series: MSCEN (left) vs MSTAGE (right)
def plot_contrast_ts(ctr_mscen, ctr_mstage, station, n_cases=3):
    tc, pc = f"{station}_true_contrast", f"{station}_pred_contrast"
    cases = sorted(ctr_mscen["case"].unique())[:n_cases]

    fig, axes = plt.subplots(n_cases, 2, figsize=(18, 4 * n_cases), constrained_layout=True)
    if n_cases == 1:
        axes = axes.reshape(1, -1)

    for i, case_id in enumerate(cases):
        for j, (ctr, label) in enumerate([(ctr_mscen, "MSCEN (shared trunk)"),
                                           (ctr_mstage, "MSTAGE (independent)")]):
            ax = axes[i, j]
            sub = ctr[ctr["case"] == case_id].sort_values("datetime")
            score = nse(sub[tc].values, sub[pc].values)
            ax.plot(sub["datetime"], sub[tc], color=C1, lw=1.5, label="True contrast")
            ax.plot(sub["datetime"], sub[pc], color=C2, lw=1.2, alpha=0.85, label="Pred contrast")
            ax.axhline(0, color="grey", lw=0.5, ls="--")
            ax.set_title(f"{label} | case={case_id} | NSE={score:.3f}", fontsize=11)
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=10)
            if i == 0 and j == 0:
                ax.legend(fontsize=9)

    fig.suptitle(f"Contrast (suisun - base) at station {station}", fontsize=14, y=1.01)
    fig.savefig(SAVE_DIR / f"fig4_contrast_ts_{station}.png", dpi=DPI)
    plt.show()

for st in ["x2", "srv", "emm2"]:
    plot_contrast_ts(ctr_mscen_suisun, ctr_mstage_suisun, st)

In [ ]:
# Slide 9: clean 1-case contrast timeseries for presentation
def plot_contrast_ts_slide(ctr_mscen, ctr_mstage, station):
    tc, pc = f'{station}_true_contrast', f'{station}_pred_contrast'
    case_id = sorted(ctr_mscen['case'].unique())[0]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True, sharey=True)
    for ax, ctr, label, color in [
        (ax1, ctr_mscen, 'Shared Trunk (MSCEN)', C1),
        (ax2, ctr_mstage, 'Independent (MSTAGE)', C2)]:
        sub = ctr[ctr['case'] == case_id].sort_values('datetime')
        score = nse(sub[tc].values, sub[pc].values)
        ax.plot(sub['datetime'], sub[tc], color='black', lw=1.3, label='True contrast')
        ax.plot(sub['datetime'], sub[pc], color=color, lw=1.1, alpha=0.85, label='Predicted')
        ax.axhline(0, color='grey', lw=0.5, ls='--')
        ax.set_title(f'{label}  |  NSE = {score:.3f}', fontsize=12)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
        ax.legend(fontsize=9, loc='lower left')
        ax.grid(alpha=0.3)
    ax1.set_ylabel('Contrast (suisun - base)', fontsize=11)
    plt.savefig(SAVE_DIR / f'fig3_contrast_ts_{station}_slide.png', dpi=DPI, bbox_inches='tight')
    plt.show()

plot_contrast_ts_slide(ctr_mscen_suisun, ctr_mstage_suisun, 'srv')

In [ ]:
# Scatter: true contrast vs predicted contrast (matched axes across both panels)
def plot_contrast_scatter(ctr_mscen, ctr_mstage, station):
    tc, pc = f"{station}_true_contrast", f"{station}_pred_contrast"
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)

    # Compute shared axis limits across BOTH panels
    all_vals = np.concatenate([
        ctr_mscen[tc].dropna().values, ctr_mscen[pc].dropna().values,
        ctr_mstage[tc].dropna().values, ctr_mstage[pc].dropna().values,
    ])
    lo, hi = np.nanmin(all_vals), np.nanmax(all_vals)
    margin = (hi - lo) * 0.1
    shared_lim = (lo - margin, hi + margin)

    for ax, ctr, label in [(ax1, ctr_mscen, "MSCEN (shared trunk)"),
                            (ax2, ctr_mstage, "MSTAGE (independent)")]:
        t, p = ctr[tc].values, ctr[pc].values
        mask = np.isfinite(t) & np.isfinite(p)
        score = nse(t, p)
        ax.scatter(t[mask], p[mask], s=6, alpha=0.35, color=C1)
        ax.plot(shared_lim, shared_lim, "k--", lw=0.8)
        ax.set_xlim(shared_lim)
        ax.set_ylim(shared_lim)
        ax.set_xlabel("True contrast")
        ax.set_ylabel("Pred contrast")
        ax.set_title(f"{label}\nNSE={score:.3f}", fontsize=11)
        ax.set_aspect("equal")
        ax.grid(alpha=0.3)

    fig.suptitle(f"Contrast scatter: station {station}", fontsize=13)
    fig.savefig(SAVE_DIR / f"fig4_scatter_{station}.png", dpi=DPI)
    plt.show()

for st in ["x2", "mrz", "srv"]:
    plot_contrast_scatter(ctr_mscen_suisun, ctr_mstage_suisun, st)

### Section 3 Experiment Log

| Label | Architecture | Pipeline | Key Params | Contrast NSE (mean, suisun) |
|-------|-------------|----------|------------|----------------------------|
| MSCEN v1 T1 | GRU(38,19) shared trunk, 3 heads | DSM2 -> multi-head | cw=0, lr=1e-3/5e-4, 45ep | +0.406 |
| MSTAGE base (RMA-1a step2) | GRU(38,19) independent | DSM2 -> rma_base | lr=1e-3/5e-4, 45ep | N/A (base only) |
| MSTAGE suisun (RMA-2a) | GRU(38,19) independent | DSM2 -> rma_suisun | lr=1e-3/5e-4, 45ep | -3.53 |

**Comparison is clean:** Same architecture (GRU 38,19), same DSM2 pretraining, same LR/epochs. Only difference is shared vs independent trunk.

**Contrast computed as:** true_contrast = suisun_ref - base_ref, pred_contrast = suisun_pred - base_pred. NSE of pred_contrast vs true_contrast across all 20 stations.

**Conclusion:** Shared trunk (MSCEN) achieves mean contrast NSE of +0.41 vs -3.53 for independent models (MSTAGE). The advantage holds across all 3 scenarios (suisun, cache, ft).

### 3.1 Why MSTAGE works at some stations

MSTAGE achieves positive contrast NSE at only 3 of 20 stations: nsl2, gzl, and vol.
These have the largest true contrast signals (contrast_std > 590).
When the contrast is large enough, even uncorrelated errors don't swamp the signal.

In [ ]:
# True contrast magnitude vs MSTAGE contrast performance
mag_rows = []
for st in STATIONS:
    tc = f'{st}_true_contrast'
    if tc not in ctr_mscen_suisun.columns: continue
    true_c = ctr_mscen_suisun[tc].dropna()
    mscen_ctr = nse(ctr_mscen_suisun[tc].values, ctr_mscen_suisun[f'{st}_pred_contrast'].values)
    mstage_ctr = nse(ctr_mstage_suisun[tc].values, ctr_mstage_suisun[f'{st}_pred_contrast'].values)
    mag_rows.append({'station': st, 'contrast_std': round(float(true_c.std()), 1),
                     'MSCEN_ctr': round(mscen_ctr, 4), 'MSTAGE_ctr': round(mstage_ctr, 4)})

df_mag = pd.DataFrame(mag_rows).sort_values('contrast_std', ascending=False)
print(df_mag.to_string(index=False))
print(f'\nStations where MSTAGE > 0: {(df_mag["MSTAGE_ctr"] > 0).sum()}/20')
print(f'These are: {df_mag[df_mag["MSTAGE_ctr"] > 0]["station"].tolist()} '
      f'with contrast_std = {df_mag[df_mag["MSTAGE_ctr"] > 0]["contrast_std"].tolist()}')
print(f'\nMSTAGE advantage only when true contrast is large enough to survive uncorrelated errors.')

## 4. Mechanism: Correlated Errors from Shared Trunk

The predicted contrast decomposes as:

```
pred_contrast = pred_scenario - pred_base
              = (true_scenario + err_scenario) - (true_base + err_base)
              = true_contrast + (err_scenario - err_base)
```

Contrast accuracy depends on the residual `(err_scenario - err_base)`:
- If errors are **correlated** (shared trunk), they cancel and the residual is small
- If errors are **uncorrelated** (independent models), `Var(err_s - err_b) = Var(err_s) + Var(err_b)` - errors compound

In [ ]:
# Per-station error correlation: MSCEN vs MSTAGE
corr_rows = []
for st in STATIONS:
    corr_rows.append(error_correlation_analysis(mscen_base, mscen_suisun, st, "MSCEN"))
    corr_rows.append(error_correlation_analysis(mstage_base, mstage_suisun, st, "MSTAGE"))

df_corr = pd.DataFrame(corr_rows)
pivot = df_corr.pivot(index="station", columns="label", values="error_corr")

# Sort by MSCEN contrast NSE (same order as section 5)
station_order = suisun["station"].values
pivot = pivot.loc[station_order]

print("Error correlation between base and scenario predictions:")
print("(Higher = errors cancel better on subtraction)\n")
print(pivot.to_string())
print(f"\nMean - MSCEN: {pivot['MSCEN'].mean():.4f}")
print(f"Mean - MSTAGE: {pivot['MSTAGE'].mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5), constrained_layout=True)

x = np.arange(len(pivot))
w = 0.35

ax.bar(x - w/2, pivot["MSCEN"].values, w, label="MSCEN (shared trunk)", color=C1, edgecolor="black")
ax.bar(x + w/2, pivot["MSTAGE"].values, w, label="MSTAGE (independent)", color=C2, edgecolor="black")

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=50, ha="right", fontsize=11)
ax.set_ylabel("Error correlation (base vs scenario)")

ax.legend(fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.grid(axis="y", alpha=0.3)

fig.savefig(SAVE_DIR / "fig5_error_correlation.png", dpi=DPI)
plt.show()

## 5. The x2 Proof: Why Good Absolute NSE Coexists with Poor Contrast NSE

Station x2 (distance to 2 ppt isohaline) is the most policy-relevant salinity metric. It illustrates the paradox clearly:
- Absolute NSE ~ 0.96 (excellent)
- Individual error std ~ 2 km, but true contrast std ~ 0.31 km (errors are 6x the signal)
- Only error correlation from the shared trunk makes contrast prediction viable at all

In [ ]:
st = "x2"
mg_mscen = pd.merge(mscen_base, mscen_suisun, on=["datetime", "case"], suffixes=("_base", "_scen"))
mg_mstage = pd.merge(mstage_base, mstage_suisun, on=["datetime", "case"], suffixes=("_base", "_scen"))

for label, mg in [("MSCEN (shared trunk)", mg_mscen), ("MSTAGE (independent)", mg_mstage)]:
    br, bp = mg[f"{st}_base"].values, mg[f"{st}_pred_base"].values
    sr, sp = mg[f"{st}_scen"].values, mg[f"{st}_pred_scen"].values
    tc, pc = sr - br, sp - bp
    err_b, err_s = bp - br, sp - sr
    err_corr = np.corrcoef(err_b, err_s)[0, 1]
    contrast_err = err_s - err_b

    base_nse_val = 1 - np.sum(err_b**2) / np.sum((br - br.mean())**2)
    suis_nse_val = 1 - np.sum(err_s**2) / np.sum((sr - sr.mean())**2)
    ctr_nse_val = 1 - np.sum((pc - tc)**2) / np.sum((tc - tc.mean())**2)

    print(f"{'=' * 65}")
    print(f"  {label}")
    print(f"{'=' * 65}")
    print(f"  Base x2:  mean={br.mean():.2f} km   std={br.std():.2f} km")
    print(f"  Base error std:     {err_b.std():.3f} km   -> NSE = {base_nse_val:.4f}")
    print(f"  Suisun error std:   {err_s.std():.3f} km   -> NSE = {suis_nse_val:.4f}")
    print(f"")
    print(f"  True contrast std:  {tc.std():.3f} km   (the signal)")
    print(f"  Individual errors:  {err_b.std():.3f} km   ({err_b.std()/tc.std():.1f}x larger)")
    print(f"")
    print(f"  Error correlation:  {err_corr:.4f}")
    print(f"  Contrast error std: {contrast_err.std():.3f} km")
    print(f"    if perfectly correlated:  0.000 km")
    print(f"    if uncorrelated:          {np.sqrt(2)*err_b.std():.3f} km")
    print(f"    actual:                   {contrast_err.std():.3f} km")
    print(f"")
    print(f"  Contrast NSE = {ctr_nse_val:.4f}")
    print()

In [ ]:
# 3-panel proof plots: absolute predictions look great, contrast does not
def plot_x2_proof(mg, label, source_tag):
    c1 = mg[mg["case"] == mg["case"].min()].sort_values("datetime")
    br, bp = c1[f"{st}_base"], c1[f"{st}_pred_base"]
    sr, sp = c1[f"{st}_scen"], c1[f"{st}_pred_scen"]
    tc, pc = sr.values - br.values, sp.values - bp.values
    dt = c1["datetime"]

    nse_b = nse(br.values, bp.values)
    nse_s = nse(sr.values, sp.values)
    nse_c = nse(tc, pc)

    fig, axes = plt.subplots(3, 1, figsize=(14, 11), constrained_layout=True)

    axes[0].plot(dt, br, color=C1, lw=1.5, label="Reference")
    axes[0].plot(dt, bp, color=C2, lw=1.2, alpha=0.85, label="Predicted")
    axes[0].set_title(f"Base scenario | NSE = {nse_b:.3f}", fontsize=12)
    axes[0].set_ylabel("x2 (km)")
    axes[0].legend()

    axes[1].plot(dt, sr, color=C1, lw=1.5, label="Reference")
    axes[1].plot(dt, sp, color=C2, lw=1.2, alpha=0.85, label="Predicted")
    axes[1].set_title(f"Suisun scenario | NSE = {nse_s:.3f}", fontsize=12)
    axes[1].set_ylabel("x2 (km)")
    axes[1].legend()

    axes[2].plot(dt, tc, color=C1, lw=1.5, label="True contrast")
    axes[2].plot(dt, pc, color=C2, lw=1.2, alpha=0.85, label="Predicted contrast")
    axes[2].axhline(0, color="grey", lw=0.5, ls="--")
    axes[2].set_title(f"Contrast (suisun - base) | NSE = {nse_c:.3f}", fontsize=12)
    axes[2].set_ylabel("x2 contrast (km)")
    axes[2].legend()

    for ax in axes:
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=10)

    fig.suptitle(f"{label} at x2 - case 1\nIndividual predictions are excellent. Contrast is not.",
                 fontsize=14, fontweight="bold")
    fig.savefig(SAVE_DIR / f"fig8_{source_tag}_x2_proof.png", dpi=DPI)
    plt.show()

plot_x2_proof(mg_mscen, "MSCEN (shared trunk)", "mscen")
plot_x2_proof(mg_mstage, "MSTAGE (independent)", "mstage")

In [ ]:
# Combined 2x3 figure: MSCEN (top) vs MSTAGE (bottom), columns = base / suisun / contrast
# Contrast column shares y-axis so MSCEN residual is visually comparable to MSTAGE.

fig, axes = plt.subplots(2, 3, figsize=(20, 10), constrained_layout=True)

# First pass: collect contrast y-range across both rows for shared ylim
all_contrast = []
for mg in [mg_mscen, mg_mstage]:
    c1 = mg[mg["case"] == mg["case"].min()].sort_values("datetime")
    br, bp = c1[f"{st}_base"].values, c1[f"{st}_pred_base"].values
    sr, sp = c1[f"{st}_scen"].values, c1[f"{st}_pred_scen"].values
    all_contrast.extend((sr - br).tolist())
    all_contrast.extend((sp - bp).tolist())
lo, hi = min(all_contrast), max(all_contrast)
margin = 0.1 * (hi - lo)
ctr_ylim = (lo - margin, hi + margin)

for row_idx, (mg, label) in enumerate([(mg_mscen, "Shared Trunk (MSCEN)"),
                                        (mg_mstage, "Independent (MSTAGE)")]):
    c1 = mg[mg["case"] == mg["case"].min()].sort_values("datetime")
    br, bp = c1[f"{st}_base"].values, c1[f"{st}_pred_base"].values
    sr, sp = c1[f"{st}_scen"].values, c1[f"{st}_pred_scen"].values
    tc, pc = sr - br, sp - bp
    dt = c1["datetime"]

    nse_b = nse(br, bp)
    nse_s = nse(sr, sp)
    nse_c = nse(tc, pc)

    ax = axes[row_idx, 0]
    ax.plot(dt, br, color=C1, lw=1.5, label="Reference")
    ax.plot(dt, bp, color=C2, lw=1.2, alpha=0.85, label="Predicted")
    ax.set_title(f"Base | NSE={nse_b:.3f}", fontsize=12)
    ax.set_ylabel(f"{label}\nx2 (km)", fontsize=11)
    if row_idx == 0:
        ax.legend(fontsize=9)

    ax = axes[row_idx, 1]
    ax.plot(dt, sr, color=C1, lw=1.5)
    ax.plot(dt, sp, color=C2, lw=1.2, alpha=0.85)
    ax.set_title(f"Suisun | NSE={nse_s:.3f}", fontsize=12)

    ax = axes[row_idx, 2]
    ax.plot(dt, tc, color=C1, lw=1.5, label="True contrast")
    ax.plot(dt, pc, color=C2, lw=1.2, alpha=0.85, label="Pred contrast")
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.set_title(f"Contrast | NSE={nse_c:.3f}", fontsize=12)
    ax.set_ylim(ctr_ylim)
    if row_idx == 0:
        ax.legend(fontsize=9)

    for col in range(3):
        axes[row_idx, col].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        axes[row_idx, col].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        plt.setp(axes[row_idx, col].xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=10)


fig.savefig(SAVE_DIR / "fig6_x2_proof_combined.png", dpi=DPI)
plt.show()


## 6. Multi-Scenario Consistency

The MSCEN v1 model predicts all 3 scenarios (suisun, cache, ft) simultaneously from a single trained model. The shared-trunk advantage holds across all scenarios.

In [ ]:
# Summary: absolute + contrast NSE for all 3 scenarios
cross_rows = []
abs_data = [
    ("suisun", df_v1.iloc[0]["mean_nse_suisun"], df_mstage_suisun.iloc[0]["mean_nse_target"]),
    ("cache", df_v1.iloc[0]["mean_nse_cache"], df_mstage_cache.iloc[0]["mean_nse_target"]),
    ("ft", df_v1.iloc[0]["mean_nse_ft"], df_mstage_ft.iloc[0]["mean_nse_target"]),
]

ctr_data = [
    ("suisun", ctr_mscen_suisun, ctr_mstage_suisun),
    ("cache", ctr_mscen_cache, ctr_mstage_cache),
    ("ft", ctr_mscen_ft, ctr_mstage_ft),
]

for i, (scenario, mscen_abs, mstage_abs) in enumerate(abs_data):
    _, ctr_ms, ctr_st = ctr_data[i]
    # Mean contrast NSE across all stations
    ms_nses, st_nses = [], []
    for st in STATIONS:
        tc, pc = f"{st}_true_contrast", f"{st}_pred_contrast"
        if tc in ctr_ms.columns:
            ms_nses.append(nse(ctr_ms[tc].values, ctr_ms[pc].values))
            st_nses.append(nse(ctr_st[tc].values, ctr_st[pc].values))

    cross_rows.append({
        "Scenario": scenario,
        "MSCEN abs NSE": round(mscen_abs, 4),
        "MSTAGE abs NSE": round(mstage_abs, 4),
        "MSCEN contrast NSE": round(np.mean(ms_nses), 4),
        "MSTAGE contrast NSE": round(np.mean(st_nses), 4),
    })

df_cross = pd.DataFrame(cross_rows)
print(df_cross.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5), constrained_layout=True)

scenarios = [s.capitalize() if s != 'ft' else 'Franks Tract' for s in df_cross['Scenario'].values]
x = np.arange(len(scenarios))
w = 0.2

ax.bar(x - 1.5*w, df_cross['MSCEN abs NSE'], w, label='Shared trunk absolute', color=C1, edgecolor='black', lw=0.8)
ax.bar(x - 0.5*w, df_cross['MSTAGE abs NSE'], w, label='Independent absolute', color=C1, alpha=0.4, edgecolor='black', lw=0.8)
ax.bar(x + 0.5*w, df_cross['MSCEN contrast NSE'], w, label='Shared trunk contrast', color=C2, edgecolor='black', lw=0.8)
mstage_ctr_clipped = df_cross['MSTAGE contrast NSE'].clip(lower=-1)
ax.bar(x + 1.5*w, mstage_ctr_clipped, w, label='Independent contrast', color=C2, alpha=0.4, edgecolor='black', lw=0.8)

for i, v in enumerate(df_cross['MSTAGE contrast NSE']):
    if v < -1:
        ax.text(x[i] + 1.5*w, -0.9, f'{v:.0f}', ha='center', fontsize=9, color=C2, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=12)
ax.set_ylabel('NSE', fontsize=12)
ax.axhline(0, color='black', lw=0.8)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=4, frameon=False, fontsize=10)
ax.set_ylim(-1, 1)
ax.grid(axis='y', alpha=0.3)
plt.savefig(SAVE_DIR / 'fig6_cross_scenario.png', dpi=DPI, bbox_inches='tight')


# SI: Supporting Information

Exploratory analyses not included in the main deck.

## SI-A: Architecture Exploration

We explored architectures beyond the simple shared-trunk MSCEN. None of the variants beat the baseline on contrast NSE.

### SI-A1: Branch Architecture Schematic

Hand-drawn on slide (no code). Compares plain MSCEN (shared trunk → Dense head) vs MSCEN-Branch (shared trunk → per-scenario GRU → Dense head).

### SI-A2: v9a 11-Trial Branch Sweep — Null

Depth view: one architecture ([32,32]+GRU(32) branch, lay2-init) tested across 11 HP combinations. All below v1 baseline on contrast NSE.

In [ ]:
# SI-A2: v9a 11-trial branch sweep null result
df_v9a = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v9a_master_results.csv")

v9a_rows = []
for _, row in df_v9a.iterrows():
    trial = row["trial_name"]
    pfx = OUTPUT / f"rma_base.multi_gru2_MSCEN_RMA_v9a_{trial}"
    try:
        mb = load_and_merge(pfx, "base")
        ms = load_and_merge(pfx, "suisun")
        ctr = build_contrast_df(mb, ms, STATIONS)
        nses = [nse(ctr[f"{st}_true_contrast"].values, ctr[f"{st}_pred_contrast"].values)
                for st in STATIONS if f"{st}_true_contrast" in ctr.columns]
        v9a_rows.append({
            "trial": trial,
            "cw": int(row["contrast_weight"]),
            "epochs": int(row["multi_main_epochs"]),
            "abs_nse": round(row["mean_nse_overall"], 4),
            "ctr_nse": round(float(np.mean(nses)), 4),
        })
    except Exception as e:
        print(f"{trial}: {e}")

df_v9a_ctr = pd.DataFrame(v9a_rows)
df_v9a_ctr = df_v9a_ctr[df_v9a_ctr["trial"] != "Trial8"].sort_values("ctr_nse").reset_index(drop=True)
df_v9a_ctr["label"] = df_v9a_ctr.apply(lambda r: f"{r['trial']}  cw={r['cw']}, {r['epochs']}ep", axis=1)

V1_CTR = 0.406

fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
y = np.arange(len(df_v9a_ctr))
ax.barh(y, df_v9a_ctr["ctr_nse"], color=C4, edgecolor="black", linewidth=0.6, zorder=3)
ax.axvline(V1_CTR, color=C3, ls="--", lw=1.8,
           label=f"v1 baseline (no branch) = {V1_CTR:.3f}", zorder=2)
ax.axvline(0, color="black", lw=0.8, alpha=0.5, zorder=1)

ax.set_yticks(y)
ax.set_yticklabels(df_v9a_ctr["label"], fontsize=9)
ax.set_xlabel("Mean Contrast NSE (suisun - base, 20 stations)")
ax.set_xlim(-0.05, 0.5)
ax.legend(loc="lower right", framealpha=0.9, fontsize=9)
ax.grid(alpha=0.3, axis="x")

for i, v in enumerate(df_v9a_ctr["ctr_nse"]):
    ax.text(v + 0.005 if v >= 0 else v - 0.005, i, f"{v:.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=8.5)

fig.text(0.01, -0.01,
         "v9a: [32,32] trunk + GRU(32) per-scenario branch, lay2-init. Trial 8 excluded (training crash).",
         fontsize=8, style="italic", color="dimgray")

fig.savefig(SAVE_DIR / "fig_SI_A2_v9a_null.png", dpi=DPI, bbox_inches="tight")
plt.show()

print(f"\n11 valid trials: max ctr NSE = {df_v9a_ctr['ctr_nse'].max():.3f}")
print(f"v1 baseline = {V1_CTR:.3f}")
print(f"All below baseline: {(df_v9a_ctr['ctr_nse'] < V1_CTR).all()}")

### SI-A4: Architecture Exploration Summary

Breadth view: best trial from each architecture variant. No variant beats v1 baseline on contrast NSE; branches sacrifice absolute accuracy.

In [ ]:
# SI-A4: best-of-each architecture (absolute + contrast)
# Compute v10 best from CSV
df_v10 = pd.read_csv(HISTORY / "gridsearch_MSCEN_RMA_v10_master_results.csv")
v10_rows = []
for _, row in df_v10.iterrows():
    trial = row["trial_name"]
    pfx = OUTPUT / f"rma_base.multi_gru2_MSCEN_RMA_v10_{trial}"
    try:
        mb = load_and_merge(pfx, "base")
        ms = load_and_merge(pfx, "suisun")
        ctr = build_contrast_df(mb, ms, STATIONS)
        nses = [nse(ctr[f"{st}_true_contrast"].values, ctr[f"{st}_pred_contrast"].values)
                for st in STATIONS if f"{st}_true_contrast" in ctr.columns]
        v10_rows.append({
            "trial": trial,
            "abs_nse": round(row["mean_nse_overall"], 4),
            "ctr_nse": round(float(np.mean(nses)), 4),
        })
    except Exception:
        pass

df_v10_ctr = pd.DataFrame(v10_rows)
v10_best = df_v10_ctr.loc[df_v10_ctr["abs_nse"].idxmax()]

arch_data = [
    {"label": "v1\n[38,19]\nno branch",            "abs": 0.9361,              "ctr": 0.406,               "group": "no branch"},
    {"label": "v6 best\n[32,16]\nno branch",       "abs": 0.9212,              "ctr": 0.382,               "group": "no branch"},
    {"label": "v10 best\n[32,16,16]\nno branch",   "abs": v10_best["abs_nse"], "ctr": v10_best["ctr_nse"], "group": "no branch"},
    {"label": "v8.1\n[16,16]+GRU(16)\nbranch",     "abs": 0.8965,              "ctr": 0.213,               "group": "branch"},
    {"label": "v9a best\n[32,32]+GRU(32)\nbranch", "abs": 0.9156,              "ctr": 0.232,               "group": "branch"},
]
df_arch = pd.DataFrame(arch_data)

V1_ABS, V1_CTR = 0.9361, 0.406
group_colors = {"no branch": C3, "branch": C4}
bar_colors = [group_colors[g] for g in df_arch["group"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
x = np.arange(len(df_arch))

ax1.bar(x, df_arch["abs"], color=bar_colors, edgecolor="black", linewidth=0.8)
ax1.axhline(V1_ABS, color=C3, ls="--", lw=1.3, alpha=0.7,
            label=f"v1 baseline = {V1_ABS:.3f}")
ax1.set_xticks(x)
ax1.set_xticklabels(df_arch["label"], fontsize=8.5)
ax1.set_ylabel("Absolute NSE")
ax1.set_ylim(0, 1.0)
ax1.grid(alpha=0.3, axis="y")
ax1.legend(loc="lower right", fontsize=9)
for i, v in enumerate(df_arch["abs"]):
    ax1.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

ax2.bar(x, df_arch["ctr"], color=bar_colors, edgecolor="black", linewidth=0.8)
ax2.axhline(V1_CTR, color=C3, ls="--", lw=1.3, alpha=0.7,
            label=f"v1 baseline = {V1_CTR:.3f}")
ax2.axhline(0, color="black", lw=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(df_arch["label"], fontsize=8.5)
ax2.set_ylabel("Contrast NSE (suisun - base)")
ax2.set_ylim(0, 0.55)
ax2.grid(alpha=0.3, axis="y")
ax2.legend(loc="lower right", fontsize=9)
for i, v in enumerate(df_arch["ctr"]):
    ax2.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=C3, edgecolor="black", label="No branch"),
                  Patch(facecolor=C4, edgecolor="black", label="Branch")]
fig.legend(handles=legend_handles, loc="upper center", ncol=2,
           bbox_to_anchor=(0.5, 1.06), framealpha=0.9, fontsize=10)

fig.savefig(SAVE_DIR / "fig_SI_A4_arch_exploration.png", dpi=DPI, bbox_inches="tight")
plt.show()

print("\nBest-of-each architecture:")
print(df_arch[["label", "abs", "ctr"]].to_string(index=False))
print(f"\nv1 baseline: abs={V1_ABS:.3f}, ctr={V1_CTR:.3f}")

## Appendix: SNR and Error Distribution at X2 (slide 11)


In [ ]:
# Signal-to-noise diagnostics at X2 — multiple framings for slide bullets

def snr_diagnostics(mg, label, station="x2"):
    br = mg[f"{station}_base"].values
    bp = mg[f"{station}_pred_base"].values
    sr = mg[f"{station}_scen"].values
    sp = mg[f"{station}_pred_scen"].values

    tc = sr - br
    err_b = bp - br
    err_s = sp - sr
    contrast_err = err_s - err_b

    signal_std = tc.std()
    abs_std_pooled = np.sqrt(0.5 * (br.var() + sr.var()))
    rmse_mean = 0.5 * (np.sqrt(np.mean(err_b**2)) + np.sqrt(np.mean(err_s**2)))
    err_std_base = err_b.std()
    ctr_err_std = contrast_err.std()
    err_corr = np.corrcoef(err_b, err_s)[0, 1]

    print(f"{'='*70}")
    print(f"  {label}   station={station}")
    print(f"{'='*70}")
    print(f"  Signal std (true contrast): {signal_std:.3f} km")
    print(f"  Abs X2 std (pooled):        {abs_std_pooled:.3f} km")
    print(f"  RMSE mean:                  {rmse_mean:.3f} km")
    print(f"  Err std (base):             {err_std_base:.3f} km")
    print(f"  Contrast residual std:      {ctr_err_std:.3f} km")
    print(f"  Error correlation:          {err_corr:.4f}")
    print(f"  Ratios:")
    print(f"    A signal/abs_std     = {signal_std/abs_std_pooled:.3f}")
    print(f"    B signal/rmse        = {signal_std/rmse_mean:.3f}")
    print(f"    C signal/err_std     = {signal_std/err_std_base:.3f}")
    print(f"    D signal/ctr_err_std = {signal_std/ctr_err_std:.3f}  (inside NSE)")
    print()

snr_diagnostics(mg_mscen,  "MSCEN (shared trunk)")
snr_diagnostics(mg_mstage, "MSTAGE (independent)")


In [ ]:
# Error distribution at X2 — percentiles to sanity-check "~2 km" claim

def error_distribution(mg, label, station="x2"):
    br = mg[f"{station}_base"].values
    bp = mg[f"{station}_pred_base"].values
    sr = mg[f"{station}_scen"].values
    sp = mg[f"{station}_pred_scen"].values

    err_b = bp - br
    err_s = sp - sr

    print(f"{'='*70}")
    print(f"  {label}   station={station}   (N={len(err_b)} timesteps)")
    print(f"{'='*70}")
    for tag, err in [("Base", err_b), ("Suisun", err_s)]:
        ae = np.abs(err)
        print(f"  {tag} scenario — absolute error distribution:")
        print(f"    MAE (mean)       : {ae.mean():.3f} km")
        print(f"    median           : {np.median(ae):.3f} km")
        print(f"    75th percentile  : {np.percentile(ae, 75):.3f} km")
        print(f"    90th percentile  : {np.percentile(ae, 90):.3f} km")
        print(f"    95th percentile  : {np.percentile(ae, 95):.3f} km")
        print(f"    99th percentile  : {np.percentile(ae, 99):.3f} km")
        print(f"    max              : {ae.max():.3f} km")
        print(f"    std of err       : {err.std():.3f} km")
        print(f"    mean bias        : {err.mean():+.3f} km")
        print()

error_distribution(mg_mscen,  "MSCEN (shared trunk)")
error_distribution(mg_mstage, "MSTAGE (independent)")


In [ ]:
# Locate the worst prediction errors at X2 — where do the 19 km misses come from?

def find_worst_errors(mg, label, station="x2", top=15):
    df = mg[["datetime", "case", f"{station}_base", f"{station}_pred_base",
             f"{station}_scen", f"{station}_pred_scen"]].copy()
    df["err_base"]   = df[f"{station}_pred_base"] - df[f"{station}_base"]
    df["err_suisun"] = df[f"{station}_pred_scen"] - df[f"{station}_scen"]
    df["abs_err_base"]   = df["err_base"].abs()
    df["abs_err_suisun"] = df["err_suisun"].abs()

    print(f"{'='*100}")
    print(f"  {label}   station={station}")
    print(f"{'='*100}")

    print(f"  Top {top} worst |error| (base scenario):")
    worst = df.nlargest(top, "abs_err_base")[[
        "datetime", "case", f"{station}_base", f"{station}_pred_base", "err_base"
    ]]
    print(worst.to_string(index=False))
    print()

    print(f"  Error breakdown by case (base scenario):")
    case_stats = df.groupby("case").agg(
        n=("err_base", "size"),
        mae=("abs_err_base", "mean"),
        p95=("abs_err_base", lambda x: np.percentile(x, 95)),
        max_err=("abs_err_base", "max"),
        std=("err_base", "std"),
    ).round(3)
    print(case_stats.to_string())
    print()

find_worst_errors(mg_mscen,  "MSCEN (shared trunk)")
find_worst_errors(mg_mstage, "MSTAGE (independent)")

# Check if worst errors are near case boundaries (antecedent window issue?)
def check_edge_effects(mg, label, station="x2", edge_days=10):
    df = mg[["datetime", "case", f"{station}_base", f"{station}_pred_base"]].copy()
    df["err"] = (df[f"{station}_pred_base"] - df[f"{station}_base"]).abs()
    df = df.sort_values(["case", "datetime"]).reset_index(drop=True)

    df["days_from_start"] = df.groupby("case").cumcount()
    df["days_from_end"] = df.groupby("case").cumcount(ascending=False)

    edge_mask = (df["days_from_start"] < edge_days) | (df["days_from_end"] < edge_days)
    print(f"{label}: {station}")
    print(f"  Edge-of-case mean |err|: {df.loc[edge_mask, 'err'].mean():.3f} km")
    print(f"  Mid-case   mean |err|:   {df.loc[~edge_mask, 'err'].mean():.3f} km")
    print(f"  Edge-of-case max |err|:  {df.loc[edge_mask, 'err'].max():.3f} km")
    print(f"  Mid-case   max |err|:    {df.loc[~edge_mask, 'err'].max():.3f} km")
    print()

check_edge_effects(mg_mscen,  "MSCEN (shared trunk)")
check_edge_effects(mg_mstage, "MSTAGE (independent)")


In [ ]:
# Inspect case 7 around the March 2011 event — is the X2 change sudden or gradual?

def plot_case_window(mg, label, station="x2", case_id=7,
                     start="2011-02-15", end="2011-05-15"):
    df = mg[mg["case"] == case_id].sort_values("datetime").copy()
    df = df[(df["datetime"] >= start) & (df["datetime"] <= end)]

    fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)
    ax.plot(df["datetime"], df[f"{station}_base"], color="black", lw=1.5, label="RMA reference")
    ax.plot(df["datetime"], df[f"{station}_pred_base"], color=C2, lw=1.2, alpha=0.85, label="ANN prediction")
    ax.axvspan(pd.Timestamp("2011-03-23"), pd.Timestamp("2011-04-07"),
               color="grey", alpha=0.2, label="Peak error window")
    ax.set_ylabel(f"{station} (km)")
    ax.set_title(f"{label} — case {case_id} — zoom on Feb-May 2011")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
    plt.show()

    # Tabular view: daily values, every 3 days
    sub = df[["datetime", f"{station}_base", f"{station}_pred_base"]].iloc[::3]
    sub["err"] = sub[f"{station}_pred_base"] - sub[f"{station}_base"]
    sub.columns = ["date", "RMA_ref", "ANN_pred", "error"]
    print(f"{'='*70}")
    print(f"  {label}  case {case_id}  Feb-May 2011 (every 3 days)")
    print(f"{'='*70}")
    print(sub.round(2).to_string(index=False))
    print()

plot_case_window(mg_mscen,  "MSCEN (shared trunk)")
plot_case_window(mg_mstage, "MSTAGE (independent)")


In [ ]:
# X2 data range — overall and per case

def x2_range(mg, label):
    br = mg["x2_base"].values
    sr = mg["x2_scen"].values
    print(f"{label}")
    print(f"  Base   — min: {br.min():.2f} km   max: {br.max():.2f} km   range: {br.max()-br.min():.2f} km   mean: {br.mean():.2f}   std: {br.std():.2f}")
    print(f"  Suisun — min: {sr.min():.2f} km   max: {sr.max():.2f} km   range: {sr.max()-sr.min():.2f} km   mean: {sr.mean():.2f}   std: {sr.std():.2f}")
    print(f"  Percentiles (base): 5%={np.percentile(br, 5):.2f}   50%={np.percentile(br, 50):.2f}   95%={np.percentile(br, 95):.2f}")
    print()

    # Per case
    df = mg[["case", "x2_base", "x2_scen"]].copy()
    case_stats = df.groupby("case").agg(
        n=("x2_base", "size"),
        min_base=("x2_base", "min"),
        max_base=("x2_base", "max"),
        mean_base=("x2_base", "mean"),
        std_base=("x2_base", "std"),
    ).round(2)
    print(f"  Per-case:")
    print(case_stats.to_string())
    print()

x2_range(mg_mscen,  "MSCEN dataset")
x2_range(mg_mstage, "MSTAGE dataset")
